Libraries Used
------------------

In [60]:
import pandas as pd
import numpy as np

billing = pd.read_excel("1_Billing.xlsx")
billing.to_csv("1_Billing.csv", index=False)

resource = pd.read_excel("2_Resource_Inventory.xlsx")
resource.to_csv("2_Resource.csv", index=False)

incidents = pd.read_excel("3_Incidents.xlsx")
incidents.to_csv("3_Incidents.csv", index=False)

tickets = pd.read_excel("4_Support_Tickets.xlsx")
tickets.to_csv("4_Support_Tickets.csv", index=False)

finance = pd.read_excel("5_Customer_Finance.xlsx")
finance.to_csv("5_Customer_Finance.csv", index=False)

security = pd.read_excel("6_Security_Logs.xlsx")
security.to_csv("6_Security_Logs.csv", index=False)

TC 1 : Account ID trimming/uppercasing and master mapping.
--------------------------------------------------------


In [61]:
billing["Account"] = billing["Account"].astype(str).str.strip().str.upper()

resource["Account"] = resource["Account"].astype(str).str.strip().str.upper()

finance["Account"] = finance["Account"].astype(str).str.strip().str.upper()

master_account = pd.concat([ billing["Account"], resource["Account"], 
                            finance["Account"]]).drop_duplicates().sort_values().reset_index(drop=True)

master_account = pd.DataFrame({ "Master_Account": master_account })

master_account.to_csv("Master_Account_Mapping.csv", index=False) # Saving the master mapping.

In [62]:
print(billing["Account"].head())  # get the frist 5 rows of account column of billing dataset.

0    ACCT1082
1    ACCT1029
2    ACCT1055
3    ACCT1065
4    ACCT1058
Name: Account, dtype: object


In [63]:
# Cecking if there is any space or lowercase character is present.
print(billing["Account"].str.contains(" ").sum())
print((billing["Account"] != billing["Account"].str.upper()).sum())

0
0


TC 2 : Timestamp Normalization
------------------------

In [64]:
# To check the columns of each dataset.

print("Billing Columns:")
print(billing.columns)

print("\nIncidents Columns:")
print(incidents.columns)

print("\nTickets Columns:")
print(tickets.columns)

print("\nSecurity Columns:")
print(security.columns)

Billing Columns:
Index(['Usage_ID', 'Account', 'Resource_ID', 'TS', 'Service', 'SKU', 'Usage',
       'Unit', 'Cost', 'Currency', 'Credit_Amount', 'Price_Per_Second',
       'Effective_Date'],
      dtype='object')

Incidents Columns:
Index(['Incident_ID', 'Resource_ID', 'Open_Time', 'Close_Time', 'Severity',
       'Status', 'Error_Count', 'SLA_Status'],
      dtype='object')

Tickets Columns:
Index(['Ticket_ID', 'Customer_ID', 'Incident_ID', 'Ticket_Description',
       'Severity', 'Category'],
      dtype='object')

Security Columns:
Index(['Log_ID', 'Resource_ID', 'Log_Source', 'Log_Timestamp',
       'Security_Event_Count', 'Carbon_Factor', 'Cloud_Provider'],
      dtype='object')


In [65]:
billing["TS"] = pd.to_datetime(billing["TS"], utc = True)  

incidents["Open_Time"] = pd.to_datetime(incidents["Open_Time"], utc = True)
incidents["Close_Time"] = pd.to_datetime(incidents["Close_Time"], utc = True)

security["Log_Timestamp"] = pd.to_datetime(security["Log_Timestamp"], utc = True)

# Saving the cleaned data.
billing.to_csv("Billing_Cleaned.csv", index=False)
incidents.to_csv("Incidents_Cleaned.csv", index=False)
tickets.to_csv("Support_Tickets_Cleaned.csv", index=False)
security.to_csv("Security_Logs_Cleaned.csv", index=False)

In [66]:
print(billing["TS"].head())

0   2026-01-05 06:00:00+00:00
1   2026-05-06 16:00:00+00:00
2   2026-01-06 02:00:00+00:00
3   2026-04-06 18:00:00+00:00
4   2026-05-19 03:00:00+00:00
Name: TS, dtype: datetime64[ns, UTC]


TC 3 : Service & SKU Canonical naming
--------------------------------------

In [67]:
print(billing.columns)

Index(['Usage_ID', 'Account', 'Resource_ID', 'TS', 'Service', 'SKU', 'Usage',
       'Unit', 'Cost', 'Currency', 'Credit_Amount', 'Price_Per_Second',
       'Effective_Date'],
      dtype='object')


In [68]:
print(billing["SKU"].unique())
print(billing["Service"].unique())

['SKU8' ' SKU19 ' 'sku8' ' SKU14 ' 'sku4' 'SKU20' ' SKU13 ' 'sku19'
 'SKU10' ' SKU9 ' 'sku7' ' SKU6 ' 'sku11' 'SKU13' 'SKU7' 'SKU5' 'SKU19'
 'SKU4' 'SKU17' 'SKU18' 'SKU9' 'SKU16' 'SKU3' 'SKU11' 'SKU14' 'SKU15'
 'SKU2' 'SKU12' 'SKU6' 'SKU1']
['DATABASE' ' Compute ' 'storage' 'NETWORK' ' Storage ' 'database'
 'STORAGE' ' Network ' 'COMPUTE' 'Database' 'Network' 'Storage' 'Compute']


In [69]:
# Generating the master SKU catalog.

sku_catalog = ( billing[["SKU", "Service"]].drop_duplicates().sort_values("SKU"))

print(sku_catalog)

        SKU    Service
7    SKU13    Compute 
4    SKU14    Storage 
1    SKU19    Compute 
13    SKU6    Storage 
10    SKU9    Network 
..      ...        ...
14    sku11    storage
8     sku19   database
5      sku4   database
11     sku7   database
2      sku8    storage

[96 rows x 2 columns]


In [70]:
# Creating Dictonary from the SKU catalog for mapping.

sku_mapping = dict(zip(sku_catalog["SKU"], sku_catalog["Service"]))

billing["Canonical_Service"] = billing["SKU"].map(sku_mapping)   # canonical service column.

sku_catalog.to_csv("SKU_Catalog.csv", index=False)

In [71]:
# Check whether the SKU have any invalid mapping or not.
invalid_sku = billing[billing["Canonical_Service"].isna()]
print(invalid_sku)

Empty DataFrame
Columns: [Usage_ID, Account, Resource_ID, TS, Service, SKU, Usage, Unit, Cost, Currency, Credit_Amount, Price_Per_Second, Effective_Date, Canonical_Service]
Index: []


TC : 4 Usage unit normalization (sec/min/hrs → seconds).
---------------------------------------------------------

In [72]:
print(billing.columns)


Index(['Usage_ID', 'Account', 'Resource_ID', 'TS', 'Service', 'SKU', 'Usage',
       'Unit', 'Cost', 'Currency', 'Credit_Amount', 'Price_Per_Second',
       'Effective_Date', 'Canonical_Service'],
      dtype='object')


In [74]:
print(billing["Unit"].unique())

['hrs ' 'Sec' 'sec']


In [73]:
# Conversion dictionary
unit_conversion = { "sec": 1, "min": 60, "hrs": 3600 }

# Convert usage to seconds
billing["Usage_Seconds"] = ( billing["Usage"] * billing["Unit"].map(unit_conversion) )

print(billing[["Usage", "Unit", "Usage_Seconds"]].head())


   Usage  Unit  Usage_Seconds
0  48698  hrs             NaN
1   6817  hrs             NaN
2   6240   Sec            NaN
3  13131   sec        13131.0
4    525   sec          525.0


TC : 5  Cost currency and decimal normalization; remove symbols.
------------------------------------------------------------------

In [75]:
billing["Cost"] = ( billing["Cost"].astype(str).str.replace(r"[₹$€£,]", "", regex=True).str.strip() ) # replacing currency suymbols and commas.

billing["Cost"] = pd.to_numeric( billing["Cost"],errors="coerce" ) ## Convert cost to numeric.

billing["Cost"] = billing["Cost"].round(2) # Round to 2 decimal places. 

In [76]:
# checking for any invalid cost  values.
invalid_cost = billing[billing["Cost"].isna()]

print(invalid_cost)

Empty DataFrame
Columns: [Usage_ID, Account, Resource_ID, TS, Service, SKU, Usage, Unit, Cost, Currency, Credit_Amount, Price_Per_Second, Effective_Date, Canonical_Service, Usage_Seconds]
Index: []


In [77]:
print(billing["Cost"].head())
print(billing["Cost"].dtype)
print(billing["Currency"].unique())

0    3895.84
1     545.36
2     499.20
3    1050.48
4      42.00
Name: Cost, dtype: float64
float64
['INR' ' INR ' 'inr']


TC : 6 Region normalization to standard slugs (ap-south-1).
-------------------------------------------------------------

In [78]:
print(resource["Region"].unique())

['EU-WEST-1' ' ap-south-1 ' 'ap-south-1' 'US-EAST-1' 'eu-west-1'
 'AP-SOUTH-1' ' us-east-1 ' 'us-east-1']


In [81]:
region_catalog = {
    "Mumbai": "ap-south-1",
    "AP South 1": "ap-south-1",
    "ap south 1": "ap-south-1",
    "ap_south_1": "ap-south-1",
    "ap-south-1": "ap-south-1",

    "Virginia": "us-east-1",
    "US East 1": "us-east-1",
    "us east 1": "us-east-1",
    "us-east-1": "us-east-1",

    "Ireland": "eu-west-1",
    "EU West 1": "eu-west-1",
    "eu west 1": "eu-west-1",
    "eu-west-1": "eu-west-1"
}

resource["Region"] = ( resource["Region"].astype(str).str.strip().replace(region_catalog) )

In [82]:
print(resource["Region"].head())
# Here we can see that this region column have the clean dataset so ,here we onli implemented the visulaization part.

0     EU-WEST-1
1    ap-south-1
2    ap-south-1
3     US-EAST-1
4    ap-south-1
Name: Region, dtype: object


TC : 7 Duplicate usage dedup by (Account, TS, SKU).
-----------------------------------------------------

In [83]:
print("Total Records Before:", len(billing)) # total nos o record before cleaning.

Total Records Before: 1001


In [84]:
duplicates = billing[ billing.duplicated( subset=["Account", "TS", "SKU"], keep=False) ]

print(duplicates)   

     Usage_ID   Account Resource_ID                        TS    Service  \
1          U2  ACCT1029    VM100072 2026-05-06 16:00:00+00:00   Compute    
1000       U2  ACCT1029    VM100072 2026-05-06 16:00:00+00:00   Compute    

          SKU  Usage  Unit    Cost Currency  Credit_Amount  Price_Per_Second  \
1      SKU19    6817  hrs   545.36     INR             0.0              0.08   
1000   SKU19    6817  hrs   545.36     INR             0.0              0.08   

     Effective_Date Canonical_Service  Usage_Seconds  
1        2026-01-01          Compute             NaN  
1000     2026-01-01          Compute             NaN  


In [85]:
billing = billing.drop_duplicates( subset=["Account", "TS", "SKU"], keep="first" )
# Removing duplicate records.

In [87]:
# Checking remaining duplicates
print( billing.duplicated( subset=["Account", "TS", "SKU"] ).sum() )


0


TC : 8  Free tier/credit adjustments tagging.
----------------------------------------------

In [88]:
print(billing["Credit_Amount"].head())

0    0.0
1    0.0
2    NaN
3    0.0
4    0.0
Name: Credit_Amount, dtype: float64


In [91]:
billing = billing.copy()

billing["Credit_Tag"] = billing["Credit_Amount"].apply( lambda x: "Credit Applied" if x < 0 else "No Credit" )

billing["Usage_Type"] = billing.apply( lambda row: "Free Tier" if abs(row["Credit_Amount"]) >= row["Cost"] and row["Credit_Amount"] < 0 
                                      else "Paid Usage", axis=1 )

In [92]:
# Result 
print( billing[ ["Cost", "Credit_Amount", "Credit_Tag", "Usage_Type"] ].head() )

      Cost  Credit_Amount Credit_Tag  Usage_Type
0  3895.84            0.0  No Credit  Paid Usage
1   545.36            0.0  No Credit  Paid Usage
2   499.20            NaN  No Credit  Paid Usage
3  1050.48            0.0  No Credit  Paid Usage
4    42.00            0.0  No Credit  Paid Usage


In [93]:
print(billing["Credit_Tag"].value_counts()) # counting the number of records for each credit tag category.

Credit_Tag
No Credit    1000
Name: count, dtype: int64


TC : 9 Anomaly detection on sudden usage spikes.
--------------------------------------------------

In [94]:
mean_usage = billing["Usage"].mean() # Mean usage 

std_usage = billing["Usage"].std() # standard deviation of usage

billing["Z_Score"] = ( (billing["Usage"] - mean_usage) / std_usage )
print(mean_usage)

print(std_usage)

25245.352
14542.98012606231


In [95]:
billing["Usage_Anomaly"] = billing["Z_Score"].apply(
    lambda x: "Spike" if abs(x) > 3 else "Normal"
)

In [96]:
# Result
print(billing[ ["Account", "Usage", "Z_Score", "Usage_Anomaly"]].head() )

    Account  Usage   Z_Score Usage_Anomaly
0  ACCT1082  48698  1.612644        Normal
1  ACCT1029   6817 -1.267165        Normal
2  ACCT1055   6240 -1.306840        Normal
3  ACCT1065  13131 -0.833003        Normal
4  ACCT1058    525 -1.699813        Normal


In [97]:
# only spikes
spikes = billing[
    billing["Usage_Anomaly"] == "Spike"
]

print(spikes)

Empty DataFrame
Columns: [Usage_ID, Account, Resource_ID, TS, Service, SKU, Usage, Unit, Cost, Currency, Credit_Amount, Price_Per_Second, Effective_Date, Canonical_Service, Usage_Seconds, Credit_Tag, Usage_Type, Z_Score, Usage_Anomaly]
Index: []


TC : 10 Tag/label normalization (owner, environment).
------------------------------------------------------

In [59]:
print(resource.columns)

Index(['Resource_ID', 'Account', 'Region', 'Owner', 'Environment',
       'CPU_Utilization', 'Memory_Utilization', 'Disk_Utilization',
       'Purchase_Type', 'Status'],
      dtype='object')


In [98]:
print(resource["Owner"].unique())

print(resource["Environment"].unique())

['DAVID' ' Alice ' nan 'CHARLIE' ' Bob ' 'david' 'bob' ' David ' 'charlie'
 'alice' 'Bob' 'Charlie' 'David' 'Alice']
['PROD' ' Dev ' 'dev' 'DEV' ' Prod ' 'qa' 'QA' 'prod' 'Dev' 'Prod']


In [100]:
# Normalize owner names.
resource["Owner"] = ( resource["Owner"].astype(str).str.strip().str.title() )

# Normalize environment values.
environment_map = {
    "prod": "Production",
    "PROD": "Production",
    "production": "Production",
    "Production": "Production",

    "dev": "Development",
    "DEV": "Development",
    "development": "Development",

    "test": "Testing",
    "TEST": "Testing",
    "testing": "Testing",

    "qa": "QA",
    "QA": "QA"
}

In [101]:
# Mapping
resource["Environment"] = ( resource["Environment"].astype(str).str.strip().replace(environment_map))

In [102]:
# After cleaning
print(resource["Owner"].unique())

print(resource["Environment"].unique())

['David' 'Alice' 'Nan' 'Charlie' 'Bob']
['Production' 'Dev' 'Development' 'Prod' 'QA']


TC : 11 Resource ID format validation and mapping to inventory.
------------------------------------------------------------------

In [103]:
print(billing.columns)

print(resource.columns)

Index(['Usage_ID', 'Account', 'Resource_ID', 'TS', 'Service', 'SKU', 'Usage',
       'Unit', 'Cost', 'Currency', 'Credit_Amount', 'Price_Per_Second',
       'Effective_Date', 'Canonical_Service', 'Usage_Seconds', 'Credit_Tag',
       'Usage_Type', 'Z_Score', 'Usage_Anomaly'],
      dtype='object')
Index(['Resource_ID', 'Account', 'Region', 'Owner', 'Environment',
       'CPU_Utilization', 'Memory_Utilization', 'Disk_Utilization',
       'Purchase_Type', 'Status'],
      dtype='object')


In [105]:
billing["Valid_Format"] = billing["Resource_ID"].str.match( r"^RES\d+$" )
# Validating resource fromat

In [107]:
# Creatig a list of valid resource ID.
inventory_ids = set(resource["Resource_ID"])

# check if resource exists.
billing["Inventory_Status"] = billing["Resource_ID"].apply( lambda x: "Mapped"
    if x in inventory_ids
    else "Not Found"
)

In [108]:
# After cleaning.
print( billing[ [ "Resource_ID","Valid_Format","Inventory_Status" ]].head())

  Resource_ID  Valid_Format Inventory_Status
0    VM100058         False           Mapped
1    VM100072         False           Mapped
2    VM100017         False           Mapped
3    VM100014         False           Mapped
4    VM100143         False           Mapped


In [109]:
# count invalid ids
print(
    billing["Inventory_Status"].value_counts()
)

Inventory_Status
Mapped    1000
Name: count, dtype: int64


TC : 12 PII masking in tickets; severity/code normalization.
---------------------------------------------------------------

In [111]:
print(tickets.columns)

Index(['Ticket_ID', 'Customer_ID', 'Incident_ID', 'Ticket_Description',
       'Severity', 'Category'],
      dtype='object')


TC : 13  Incident linkage to affected resources/time windows.
---------------------------------------------------------------

In [113]:
# Convert Timestamp Columns to Datetime
billing["TS"] = pd.to_datetime(billing["TS"])

incidents["Open_Time"] = pd.to_datetime(incidents["Open_Time"])

incidents["Close_Time"] = pd.to_datetime(incidents["Close_Time"])

In [115]:
# Create an Incident Time Window
incidents["Start_Time"] = incidents["Open_Time"] - pd.Timedelta(minutes=30)

incidents["End_Time"] = incidents["Close_Time"] + pd.Timedelta(minutes=30)

In [116]:
# Merge Billing with Incidents Using Resource_ID
merged = billing.merge(
    incidents,
    on="Resource_ID",
    how="left",
    suffixes=("_Billing", "_Incident")
)

In [118]:
# Check if Billing Timestamp Falls Inside the Incident Window
merged["Incident_Linked"] = (
    (merged["TS"] >= merged["Start_Time"]) &
    (merged["TS"] <= merged["End_Time"])
)

merged["Incident_Status"] = merged["Incident_Linked"].map({
    True: "Affected",
    False: "Not Affected"
})


In [121]:
# After Cleaning
print(
    merged[
        [
            "Account",
            "Resource_ID",
            "TS",
            "Start_Time",
            "End_Time",
            "Incident_Status"
        ]
    ].head()
)

    Account Resource_ID                        TS                Start_Time  \
0  ACCT1082    VM100058 2026-01-05 06:00:00+00:00                       NaT   
1  ACCT1029    VM100072 2026-05-06 16:00:00+00:00                       NaT   
2  ACCT1055    VM100017 2026-01-06 02:00:00+00:00                       NaT   
3  ACCT1065    VM100014 2026-04-06 18:00:00+00:00                       NaT   
4  ACCT1058    VM100143 2026-05-19 03:00:00+00:00 2026-02-04 23:30:00+00:00   

                   End_Time Incident_Status  
0                       NaT    Not Affected  
1                       NaT    Not Affected  
2                       NaT    Not Affected  
3                       NaT    Not Affected  
4 2026-02-05 01:56:00+00:00    Not Affected  


In [122]:
# Count 
print(merged["Incident_Status"].value_counts())


Incident_Status
Not Affected    1047
Affected           1
Name: count, dtype: int64


TC : 14 SKU price list versioning and effective dating.
----------------------------------------------------------

In [124]:
# Converting the Billing Dataset 
billing["TS"] = pd.to_datetime(billing["TS"])

In [125]:
# Creating price list table
price_catalog = pd.DataFrame({
    "SKU": [
        "SKU1","SKU1","SKU1",
        "SKU2","SKU2",
        "SKU3","SKU3"
    ],

    "Effective_Date": [
        "2025-01-01",
        "2025-04-01",
        "2025-07-01",

        "2025-01-01",
        "2025-06-01",

        "2025-01-01",
        "2025-05-01"
    ],

    "Unit_Price":[
        0.10,
        0.12,
        0.15,

        0.20,
        0.25,

        0.30,
        0.35
    ]
})

price_catalog["Effective_Date"] = pd.to_datetime(
    price_catalog["Effective_Date"]
)

In [134]:
# Same datetime type
billing["TS"] = pd.to_datetime(
    billing["TS"]
).dt.tz_localize(None)

price_catalog["Effective_Date"] = pd.to_datetime(
    price_catalog["Effective_Date"]
)

# Sort both DataFrames
billing = billing.sort_values("TS").reset_index(drop=True)

price_catalog = price_catalog.sort_values("Effective_Date").reset_index(drop=True)

# Merge
billing = pd.merge_asof(
    billing,
    price_catalog,
    left_on="TS",
    right_on="Effective_Date",
    by="SKU",
    direction="backward"
)

In [136]:
# After cleaning
print(
    billing[
        [
            "SKU",
            "TS",
            "Unit_Price"
        ]
    ].head()
)

     SKU                  TS  Unit_Price
0  SKU12 2026-01-01 06:00:00         NaN
1  SKU15 2026-01-01 10:00:00         NaN
2   SKU8 2026-01-01 10:00:00         NaN
3  SKU20 2026-01-01 10:00:00         NaN
4   SKU4 2026-01-01 11:00:00         NaN


TC : 15 Cross‑account consolidation and FX conversion (if multi‑curr).
----------------------------------------------------------------------

In [137]:
print(billing["Currency"].unique())

['INR' 'inr' ' INR ']


In [138]:
# Conversion table
fx_rates = {
    "USD":1.00,
    "INR":0.012,
    "EUR":1.09,
    "GBP":1.27
}

In [139]:
# Map exchange rates
billing["FX_Rate"] = billing["Currency"].map(fx_rates)

In [140]:
# Convert Cost to USD
billing["Cost_USD"] = ( billing["Cost"] * billing["FX_Rate"] ).round(2)

In [141]:
account_summary = (
    billing.groupby("Account", as_index=False)
    .agg(
        Total_Cost_USD=("Cost_USD", "sum"),
        Total_Usage=("Usage", "sum"),
        Total_Transactions=("Account", "count")
    )
)

In [142]:
print(account_summary.head())

    Account  Total_Cost_USD  Total_Usage  Total_Transactions
0  ACCT1001          287.41       299384                  12
1  ACCT1002          346.93       361389                  10
2  ACCT1003          204.59       213118                   8
3  ACCT1004          229.51       239085                   7
4  ACCT1005          343.57       357881                  13


TC : 16 Idle/underutilized resource detection rules.
-----------------------------------------------------

In [143]:
print(resource.columns)

Index(['Resource_ID', 'Account', 'Region', 'Owner', 'Environment',
       'CPU_Utilization', 'Memory_Utilization', 'Disk_Utilization',
       'Purchase_Type', 'Status'],
      dtype='object')


In [144]:
print(
    resource[
        [
            "Resource_ID",
            "CPU_Utilization",
            "Memory_Utilization"
        ]
    ].head()
)

  Resource_ID  CPU_Utilization  Memory_Utilization
0    VM100001               42                  19
1    VM100002              120                  -5
2    VM100003               74                  87
3    VM100004              120                  -5
4    VM100005               63                  15


In [145]:
def utilization_status(cpu):

    if cpu < 5:
        return "Idle"

    elif cpu < 20:
        return "Underutilized"

    else:
        return "Active"

In [146]:
resource["Utilization_Status"] = (
    resource["CPU_Utilization"]
    .apply(utilization_status)
)

TC : 17  Reserved/spot vs on‑demand flag normalization.
---------------------------------------------------------

In [152]:
print(billing.columns)

Index(['Usage_ID', 'Account', 'Resource_ID', 'TS', 'Service', 'SKU', 'Usage',
       'Unit', 'Cost', 'Currency', 'Credit_Amount', 'Price_Per_Second',
       'Effective_Date_x', 'Canonical_Service', 'Usage_Seconds', 'Credit_Tag',
       'Usage_Type', 'Z_Score', 'Usage_Anomaly', 'Valid_Format',
       'Inventory_Status', 'Effective_Date_y', 'Unit_Price', 'FX_Rate',
       'Cost_USD'],
      dtype='object')


In [157]:
billing = billing.merge(
    resource[["Resource_ID", "Purchase_Type"]],
    on="Resource_ID",
    how="left"
)

In [154]:
pricing_map = {
    "on demand": "On-Demand",
    "On Demand": "On-Demand",
    "ON DEMAND": "On-Demand",
    "ON_DEMAND": "On-Demand",
    "On-Demand": "On-Demand",

    "reserved": "Reserved",
    "Reserved": "Reserved",
    "Reserved Instance": "Reserved",
    "RI": "Reserved",

    "spot": "Spot",
    "Spot": "Spot",
    "SPOT": "Spot",
    "Spot VM": "Spot"
}

In [164]:
billing["Purchase_Type"] = (
    billing["Purchase_Type"]
    .astype(str)
    .str.strip()
    .replace(pricing_map)
)

In [165]:
# Verify
print(billing["Purchase_Type"].unique())

['RESERVED' 'Spot' 'On-Demand' 'on_demand' 'Reserved']


In [167]:
# Count
print(
    billing["Purchase_Type"]
    .value_counts()
)

Purchase_Type
On-Demand    368
Spot         326
RESERVED     298
Reserved       5
on_demand      3
Name: count, dtype: int64


TC : 18 Cost allocation key validation (dept/project).
-------------------------------------------------------

In [ ]:
print(billing.columns)

Index(['Usage_ID', 'Account', 'Resource_ID', 'TS', 'Service', 'SKU', 'Usage',
       'Unit', 'Cost', 'Currency', 'Credit_Amount', 'Price_Per_Second',
       'Effective_Date_x', 'Canonical_Service', 'Usage_Seconds', 'Credit_Tag',
       'Usage_Type', 'Z_Score', 'Usage_Anomaly', 'Valid_Format',
       'Inventory_Status', 'Effective_Date_y', 'Unit_Price', 'FX_Rate',
       'Cost_USD', 'Purchase_Type_x', 'Purchase_Type_y', 'Purchase_Type'],
      dtype='object')


In [170]:
print(resource.columns)

Index(['Resource_ID', 'Account', 'Region', 'Owner', 'Environment',
       'CPU_Utilization', 'Memory_Utilization', 'Disk_Utilization',
       'Purchase_Type', 'Status', 'Utilization_Status'],
      dtype='object')


TC : 19 . SLA event marking from status pages.
-----------------------------------------------

In [ ]:


incidents["SLA_Status"] = (
    incidents["SLA_Status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

sla_map = {
    "MET": "SLA Met",
    "BREACHED": "SLA Breached",
    "WARNING": "SLA Warning"
}

incidents["SLA_Event"] = incidents["SLA_Status"].replace(sla_map)

print(incidents[["SLA_Status", "SLA_Event"]].head())

  SLA_Status     SLA_Event
0   BREACHED  SLA Breached
1        MET       SLA Met
2        MET       SLA Met
3   BREACHED  SLA Breached
4   BREACHED  SLA Breached


In [173]:
# Calculate SAL from timestamps.
incidents["Open_Time"] = pd.to_datetime(incidents["Open_Time"])
incidents["Close_Time"] = pd.to_datetime(incidents["Close_Time"])

incidents["Resolution_Hours"] = (
    incidents["Close_Time"] - incidents["Open_Time"]
).dt.total_seconds() / 3600

In [174]:
incidents["SLA_Event"] = incidents["Resolution_Hours"].apply(
    lambda x: "SLA Met" if x <= 4 else "SLA Breached"
)

In [175]:
incidents["SLA_Event"] = incidents.apply(
    lambda row:
        "Open Incident"
        if row["Status"] == "Open"
        else (
            "SLA Met"
            if row["SLA_Status"] == "Met"
            else "SLA Breached"
        ),
    axis=1
)

TC : 20  Log time skew correction across sources.
--------------------------------------------------

In [176]:
# Convert timestamps
security["Log_Timestamp"] = pd.to_datetime(
    security["Log_Timestamp"],
    errors="coerce"
)

In [177]:
# Example source-specific skew (in minutes)
time_skew = {
    "AWS": 0,
    "AZURE": -5,
    "GCP": 2
}

In [178]:
# Standardize source names
security["Log_Source"] = (
    security["Log_Source"]
    .str.strip()
    .str.upper()
)

In [179]:
# Apply correction
security["Corrected_Time"] = security.apply(
    lambda row: row["Log_Timestamp"] +
    pd.Timedelta(minutes=time_skew.get(row["Log_Source"], 0)),
    axis=1
)

In [180]:
print(security[["Log_Source", "Log_Timestamp", "Corrected_Time"]].head())

    Log_Source             Log_Timestamp            Corrected_Time
0   CLOUDWATCH 2026-01-01 01:00:00+00:00 2026-01-01 01:00:00+00:00
1  STACKDRIVER 2026-01-01 02:00:00+00:00 2026-01-01 02:00:00+00:00
2  STACKDRIVER 2026-01-01 03:00:00+00:00 2026-01-01 03:00:00+00:00
3   CLOUDWATCH 2026-01-01 04:00:00+00:00 2026-01-01 04:00:00+00:00
4  STACKDRIVER 2026-01-01 05:00:00+00:00 2026-01-01 05:00:00+00:00
